# PDF Workflow: Scan -> Merge -> Split -> CSV -> Rename
Dieses Notebook bildet den kompletten Workflow in klaren Schritten ab.
Passe im ersten Code-Block nur die Parameter an und führe danach die Abschnitte nacheinander aus.

In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

# === Parameter ===
RAW_SCANS = Path(r"C:\Users\alexsc31\Documents_privat\FH Campus\Clinical_Engineering\CE28-2026\Physik\Kurztests\Kurztest_3_v2\RAW_SCANS")
PAGES_PER_SPLIT = 2
STUDENT_LIST_CSV = Path(r"C:\Users\alexsc31\Documents_privat\FH Campus\Clinical_Engineering\CE28-2026\Physik\CE_28_Studierendenliste_GHKLS_SS2026.csv")

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Code").is_dir() and (candidate / "Code" / "merge_pdfs.py").is_file():
            return candidate
    raise RuntimeError("Konnte Repo-Root mit Code-Skripten nicht finden. Notebook bitte im Repo starten.")

REPO_ROOT = find_repo_root(Path.cwd())
CODE_DIR = REPO_ROOT / "Code"

def run_script(script_name: str, *args: str) -> None:
    script_path = CODE_DIR / script_name
    if not script_path.is_file():
        raise FileNotFoundError(f"Skript nicht gefunden: {script_path}")

    command = [sys.executable, str(script_path), *map(str, args)]
    print("\n>", " \
,
,
,
Python: {sys.executable}")
print(f"Repo-Root: {REPO_ROOT}")
print(f"RAW_SCANS: {RAW_SCANS}")
print(f"Studentenliste: {STUDENT_LIST_CSV}")

## 1) RAW_SCANS ist der Ordner mit den gescannten PDFs
Hier wird geprüft, ob der Ordner existiert und PDF-Dateien enthält.

In [ ]:
if not RAW_SCANS.is_dir():
    raise FileNotFoundError(f"RAW_SCANS existiert nicht: {RAW_SCANS}")

raw_pdfs = sorted(RAW_SCANS.glob("*.pdf"))
if not raw_pdfs:
    raise RuntimeError(f"Keine PDFs in RAW_SCANS gefunden: {RAW_SCANS}")

print(f"Gefundene PDFs in RAW_SCANS: {len(raw_pdfs)}")
for pdf in raw_pdfs[:10]:
    print(" -", pdf.name)
if len(raw_pdfs) > 10:
    print(" - ...")

## 2) Erstelle "merged" und merge alle PDFs
Es wird der Ordner `RAW_SCANS/merged` angelegt und dort `merged.pdf` erzeugt.

In [ ]:
merged_dir = RAW_SCANS / "merged"
merged_dir.mkdir(parents=True, exist_ok=True)
merged_pdf = merged_dir / "merged.pdf"

run_script(
    "merge_pdfs.py",
    str(RAW_SCANS),
    "--output-dir", str(merged_dir),
    "--output-name", merged_pdf.name,
)

print(f"Merge abgeschlossen: {merged_pdf}")

## 3) Split nach jeder N-ten Seite
Das gemergte Dokument wird in `RAW_SCANS/merged/split` in Teil-PDFs zerlegt.

In [ ]:
split_dir = merged_dir / "split"
split_dir.mkdir(parents=True, exist_ok=True)

run_script(
    "split_pdfs.py",
    str(merged_dir),
    "--pages-per-file", str(PAGES_PER_SPLIT),
    "--output-dir", str(split_dir),
)

split_pdfs = sorted(split_dir.glob("*.pdf"))
print(f"Split abgeschlossen: {len(split_pdfs)} Dateien in {split_dir}")

## 4) Erstelle CSV mit current_name und leerem new_name
Es wird `pdf_names.csv` in `RAW_SCANS/merged/split` erzeugt.

In [ ]:
pdf_names_csv = split_dir / "pdf_names.csv"

run_script(
    "export_pdf_names_to_csv.py",
    str(split_dir),
    "--csv-file", str(pdf_names_csv),
)

print(f"CSV erstellt: {pdf_names_csv}")

## 5) Manueller Schritt: Kürzel eintragen
Öffne `pdf_names.csv` und trage die Kürzel in die Spalte `new_name` ein.
Führe danach die nächste Zelle aus.

In [ ]:
if not pdf_names_csv.is_file():
    raise FileNotFoundError(f"Datei fehlt: {pdf_names_csv}")

print("Bitte jetzt Kürzel in pdf_names.csv eintragen und dann mit Schritt 6 fortfahren.")
print(pdf_names_csv)

## 6) Nachnamen an Kürzel anhängen (append_student_*)
Das Skript ergänzt die eingetragenen Kürzel zu `KÜRZEL_Nachname` und meldet unscharfe Treffer.

In [ ]:
pdf_names_with_names_csv = split_dir / "pdf_names_with_names.csv"

run_script(
    "append_student_surnames_to_csv.py",
    "--input-csv", str(pdf_names_csv),
    "--code-column", "new_name",
    "--output-column", "new_name",
    "--output-csv", str(pdf_names_with_names_csv),
    "--student-list", str(STUDENT_LIST_CSV),
)

print(f"Erweiterte CSV erstellt: {pdf_names_with_names_csv}")

## 7) PDFs in `rename` umbenennen (Originale bleiben in `split`)
Die Dateien werden zuerst nach `RAW_SCANS/merged/split/rename` kopiert und dort mit der fertigen CSV umbenannt.

In [ ]:
rename_dir = split_dir / "rename"
rename_dir.mkdir(parents=True, exist_ok=True)

# PDFs in rename-Ordner kopieren (Originale in split bleiben unverändert)
copied = 0
for pdf_path in sorted(split_dir.glob("*.pdf")):
    if pdf_path.parent == rename_dir:
        continue
    target = rename_dir / pdf_path.name
    shutil.copy2(pdf_path, target)
    copied += 1

rename_csv = rename_dir / "pdf_names.csv"
shutil.copy2(pdf_names_with_names_csv, rename_csv)

run_script(
    "rename_pdfs_from_csv.py",
    str(rename_dir),
    "--csv-file", str(rename_csv),
    "--apply",
)

print(f"Kopierte PDFs: {copied}")
print(f"Umbenannte PDFs liegen in: {rename_dir}")
print(f"Originale bleiben in: {split_dir}")